# IGCD End-to-End Pipeline

This single Colab notebook orchestrates the complete Indian Glacier Change Dataset workflow. Core implementation remains in the reusable `igcd/` Python package.

## Important Execution Model

Google Earth Engine Drive exports are asynchronous. Run the inventory and export-submission stages first, wait for Earth Engine tasks to finish, then rerun this notebook with `RUN_SYNC_EXPORTS`, `RUN_CHANGE_MASKS`, `RUN_QC`, and `RUN_PACKAGING` enabled.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    %pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/IGCD')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DRIVE_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
print({'project_root': str(PROJECT_ROOT), 'drive_root': str(DRIVE_ROOT)})

In [ ]:
RUN_SETUP = True
RUN_ENVIRONMENT = True
RUN_INVENTORY = True
RUN_SENTINEL_EXPORTS = True
RUN_MASK_EXPORTS = False
RUN_SYNC_EXPORTS = False
RUN_CHANGE_MASKS = False
RUN_QC = False
RUN_PACKAGING = False

# Keep this at 1500 for two years: 1500 t1 + 1500 t2 = 3000 tasks.
MAX_GLACIERS_FOR_RUN = 1500
EXPORT_YEARS = None
MAX_EXPORT_TASKS_PER_RUN = 3000

In [ ]:
import json
import platform

import geopandas as gpd
import pandas as pd
import rasterio
from tqdm.auto import tqdm

from igcd.change_detection import generate_change_mask, summarize_change_mask
from igcd.config import load_config, write_default_config
from igcd.drive_sync import organize_drive_exports
from igcd.ee_utils import authenticate_and_initialize, validate_ee_access
from igcd.glacier_delineation import export_delineation_for_inventory
from igcd.glacier_inventory import export_inventory, prepare_inventory_from_ee
from igcd.io import write_single_band_raster
from igcd.packaging import package_dataset
from igcd.quality_control import (
    validate_alignment,
    validate_nonempty_mask,
    validate_raster,
    write_quality_reports,
)
from igcd.sentinel import export_sentinel_for_inventory
from igcd.utils import ensure_directories, setup_logging, write_json
from igcd.visualization import plot_inventory_overview, save_change_preview

In [ ]:
if RUN_SETUP:
    folders = [
        'config', 'igcd', 'notebooks', 'tests', 'logs', 'reports',
        'data/raw', 'data/interim', 'data/processed', 'exports',
        'dataset/images', 'dataset/labels', 'dataset/metadata'
    ]
    ensure_directories(PROJECT_ROOT / folder for folder in folders)
    config_path = PROJECT_ROOT / 'config' / 'config.json'
    if not config_path.exists():
        write_default_config(config_path)
    print('Project structure is ready.')

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
MAX_GLACIERS_FOR_RUN = config.raw.get('processing', {}).get(
    'max_glaciers_per_export_batch', MAX_GLACIERS_FOR_RUN
)
MAX_EXPORT_TASKS_PER_RUN = config.raw['export'].get(
    'max_tasks_per_run', MAX_EXPORT_TASKS_PER_RUN
)
logger = setup_logging(config.paths['logs'])
print({'dataset': config.dataset_name, 'years': config.years})

In [ ]:
if RUN_ENVIRONMENT:
    authenticate_and_initialize(project=config.raw['earth_engine']['project'])
    statuses = validate_ee_access(config)
    report = {
        'python': sys.version,
        'platform': platform.platform(),
        'earth_engine_project': config.raw['earth_engine']['project'],
        'datasets': [status.__dict__ for status in statuses],
    }
    write_json(report, config.paths['reports'] / 'environment_report.json')
    report

In [ ]:
if RUN_INVENTORY:
    inventory = prepare_inventory_from_ee(config, logger)
    if MAX_GLACIERS_FOR_RUN is not None:
        inventory = inventory.head(int(MAX_GLACIERS_FOR_RUN)).copy()
    outputs = export_inventory(inventory, config.paths['processed'] / 'inventory')
    plot_inventory_overview(
        inventory,
        config.paths['reports'] / 'inventory_overview.png',
    )
else:
    inventory = gpd.read_file(
        config.paths['processed'] / 'inventory' / 'glacier_inventory.geojson'
    )
    if MAX_GLACIERS_FOR_RUN is not None:
        inventory = inventory.head(int(MAX_GLACIERS_FOR_RUN)).copy()

inventory_for_ee = inventory.copy()
inventory_for_ee['geometry'] = inventory_for_ee.geometry.apply(
    lambda geom: geom.__geo_interface__
)
print({'inventory_records': len(inventory_for_ee)})

In [ ]:
if RUN_SENTINEL_EXPORTS:
    sentinel_manifest = export_sentinel_for_inventory(
        inventory_for_ee,
        config,
        logger,
        config.paths['reports'] / 'sentinel_acquisition_manifest.csv',
        years=EXPORT_YEARS,
        max_tasks=MAX_EXPORT_TASKS_PER_RUN,
    )
    display(sentinel_manifest.head())

In [ ]:
if RUN_MASK_EXPORTS:
    mask_manifest = export_delineation_for_inventory(
        inventory_for_ee,
        config,
        logger,
        config.paths['reports'] / 'glacier_delineation_manifest.csv',
        years=EXPORT_YEARS,
        max_tasks=MAX_EXPORT_TASKS_PER_RUN,
    )
    display(mask_manifest.head())

In [ ]:
if RUN_SYNC_EXPORTS:
    drive_export_dir = DRIVE_ROOT / config.raw['export']['drive_folder']
    organized = organize_drive_exports(drive_export_dir, config, logger)
    print({'organized_files': len(organized), 'drive_export_dir': str(drive_export_dir)})

In [ ]:
if RUN_CHANGE_MASKS:
    base_year = config.baseline_year
    target_year = config.target_year
    mask_root = config.paths['processed'] / 'masks'
    change_root = config.paths['processed'] / 'change_masks'
    preview_root = config.paths['reports'] / 'change_previews'
    stats = []

    for base_path in tqdm(sorted((mask_root / str(base_year)).glob('*_mask.tif'))):
        glacier_id = '_'.join(base_path.stem.split('_')[:2])
        target_path = mask_root / str(target_year) / f'{glacier_id}_{target_year}_mask.tif'
        if not target_path.exists():
            logger.warning('Missing target mask for %s', glacier_id)
            continue
        with rasterio.open(base_path) as src:
            baseline = src.read(1)
            profile = src.profile
            pixel_area = abs(src.transform.a * src.transform.e)
        with rasterio.open(target_path) as src:
            target = src.read(1)
        change = generate_change_mask(baseline, target)
        output = change_root / f'{glacier_id}_{base_year}_{target_year}_change.tif'
        write_single_band_raster(change, profile, output)
        save_change_preview(change, preview_root / f'{glacier_id}_change.png')
        summary = summarize_change_mask(change, pixel_area).__dict__
        summary.update({'glacier_id': glacier_id, 'change_mask': str(output)})
        stats.append(summary)

    stats_df = pd.DataFrame(stats)
    stats_df.to_csv(config.paths['reports'] / 'change_statistics.csv', index=False)
    display(stats_df.head())

In [ ]:
if RUN_QC:
    records = []
    for path in sorted(config.paths['processed'].glob('**/*.tif')):
        records.extend(validate_raster(path))
    for mask_path in sorted((config.paths['processed'] / 'masks').glob('*/*_mask.tif')):
        records.append(
            validate_nonempty_mask(
                mask_path,
                config.raw['quality_control']['min_mask_pixels'],
            )
        )
        glacier_id = '_'.join(mask_path.stem.split('_')[:2])
        year = mask_path.parent.name
        image = config.paths['exports'] / year / f'{glacier_id}_{year}_sentinel.tif'
        if image.exists():
            records.append(validate_alignment(image, mask_path))

    outputs = write_quality_reports(
        records,
        config.paths['reports'] / 'quality_report.csv',
        config.paths['reports'] / 'quality_summary.json',
    )
    print(outputs)

In [ ]:
if RUN_PACKAGING:
    inventory = gpd.read_file(
        config.paths['processed'] / 'inventory' / 'glacier_inventory.geojson'
    )
    image_paths = {}
    for path in sorted(config.paths['exports'].glob('*/*_sentinel.tif')):
        parts = path.stem.split('_')
        image_paths[('_'.join(parts[:2]), int(parts[2]))] = path

    label_paths = {}
    for path in sorted((config.paths['processed'] / 'change_masks').glob('*_change.tif')):
        glacier_id = '_'.join(path.stem.split('_')[:2])
        label_paths[glacier_id] = path

    metadata = {
        'name': config.dataset_name,
        'version': config.raw['dataset']['version'],
        'years': config.years,
        'source_collections': config.raw['earth_engine'],
        'export': config.raw['export'],
    }
    outputs = package_dataset(
        inventory,
        image_paths,
        label_paths,
        config.paths['dataset'],
        config.raw['splits'],
        metadata,
    )
    print(outputs)